# Day 1(M1 Day01) 실습 — 단일 vs 멀티 에이전트 협업 비교

**목표**: 규칙이 많은 작업에서 단일 에이전트의 한계를 확인하고, 역할을 나눈 협업 파이프라인과 품질·비용·속도를 숫자로 비교한다.
## 오늘의 전체 그림

**오늘의 질문**: 일을 나누면 결과가 좋아지나? 좋아진다면 대가는 얼마인가?

```text
[Part 1 만든다]   검사기 check_rules · 단일 run_single · 협업 run_multi
      │           → 1회 비교: 협업이 위반을 줄인다
      ▼
[Part 2 바꿔보기]   같은 함수로 조건만 바꾼다 (규칙 수 · 자기 검토 · LLM 검토자 · 반복 · 비용)
      │           → 협업이 이기는 조건과 대가 + (확장) 루프 run_multi_loop
      ▼
[Part 3 판단한다] 단일 · 협업 · 루프 × 상품 3종 × 3회
                  → 평균표 → 결론 3줄 리포트 + (확장) run_pipeline으로 다른 도메인
```

| Part | 블록 | 답하는 질문 | 다음 Part로 넘기는 것 |
| --- | --- | --- | --- |
| Part 1 | 따라 하기 | 일을 나누면 위반이 줄어드나? | `check_rules` · `run_single` · `run_multi` |
| Part 2 | 바꿔 보기 | 협업은 항상 좋을까? 대가는? | 비교할 방식 3종 (단일 · 협업 · 루프) |
| Part 3 | 미니 프로젝트 | 언제 무엇을 쓸 것인가? | `run_pipeline` → Day 2 역할 설계 |


## 0. 환경 준비

> 💡 오늘은 프레임워크 없이 **Python 함수**로 에이전트를 짠다. LangChain은 모델 호출(`ChatOpenAI`)에만 쓴다 → 협업 파이프라인이 결국 "함수를 부르는 순서"임을 먼저 본다. 다음 일을 동적으로 정해야 하는 Day 3(Supervisor)에서 LangGraph로 옮긴다.

모델 객체를 **두 개** 만든다. 역할이 다르면 설정도 다르다.

| 객체 | temperature | 이유 |
| --- | --- | --- |
| `llm_writer` (작성자용) | 0.9 | 매번 다른 초안 → 실험이 된다 |
| `llm_fixer` (수정자·검토자용) | 0 | 지시를 흔들림 없이 수행 |

<details>
<summary>용어 미리 풀기 — temperature</summary>

- **temperature**: 답의 다양성을 조절하는 값이다. 0이면 거의 같은 답, 높으면 매번 다른 답이 나온다.
- 작성자는 초안이 매번 달라야 "1회 운"이 아닌 경향을 볼 수 있고, 수정자는 같은 지적에 같은 수정을 해야 한다.
</details>

In [1]:
import os

# 추적 끄기: 2과목 Day17에서 .env의 LANGSMITH_TRACING을 켰다면 오늘 호출이 전부 전송된다
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

import re
import time

import pandas as pd
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm_writer = ChatOpenAI(model="gpt-4o-mini", temperature=0.9)  # 작성자: 다양한 초안
llm_fixer = ChatOpenAI(model="gpt-4o-mini", temperature=0)  # 수정자·검토자: 정확한 수행

print("준비 완료:", llm_writer.model_name)

준비 완료: gpt-4o-mini


## Part 1. 따라 하기 — 검사기와 협업 파이프라인

**전체 그림 1/3**: 검사기 · 단일 · 협업 함수를 만든다 → Part 2·3이 이 함수를 그대로 다시 쓴다.

완성 코드를 따라 치며 "작성자 → 검사기 → 수정자" 파이프라인을 만든다.

### 1-1. 과제 정의 — 상품 정보와 규칙 8개

현실의 AI 서비스에는 지켜야 할 규칙이 많다 (광고 심의·브랜드 톤·형식). 오늘 과제는 규칙 8개가 붙은 상품 소개문 작성이다.

<details>
<summary>규칙을 이렇게 설계한 이유</summary>

- 모든 규칙은 **코드로 직접 판정**할 수 있다 (문장 수·포함 여부·금지어·기호).
- 사람 취향이 아니라 코드가 채점한다 → 몇 번을 실행해도 기준이 같다.
</details>

In [2]:
# 상품 정보
product = {
    "name": "숲속 캠핑 의자",
    "price": "29,900원",
    "features": ["가벼워서 휴대하기 편함", "접이식 디자인", "컵홀더 포함", "야외 활동에 적합"],
}

# 상품 소개 규칙 — {name}·{price} 자리에 상품 정보를 채운다
PRODUCT_INTRO_RULES = """
[작성 규칙]
1. 정확히 3문장으로 쓴다. 마침표는 정확히 3개 사용한다.
2. 제품명 "{name}"을 그대로 포함한다.
3. 가격 "{price}"를 그대로 포함한다.
4. "최고", "완벽", "혁신", "프리미엄", "특별"을 사용하지 않는다.
5. 느낌표와 물음표를 사용하지 않는다.
6. 쉼표는 가격 표기 안에서만 사용한다.
7. 마지막 문장은 정확히 "지금 만나보세요."로 작성한다.
8. 가격 외의 숫자는 사용하지 않는다.
"""

print(PRODUCT_INTRO_RULES.format(name=product["name"], price=product["price"]))


[작성 규칙]
1. 정확히 3문장으로 쓴다. 마침표는 정확히 3개 사용한다.
2. 제품명 "숲속 캠핑 의자"을 그대로 포함한다.
3. 가격 "29,900원"를 그대로 포함한다.
4. "최고", "완벽", "혁신", "프리미엄", "특별"을 사용하지 않는다.
5. 느낌표와 물음표를 사용하지 않는다.
6. 쉼표는 가격 표기 안에서만 사용한다.
7. 마지막 문장은 정확히 "지금 만나보세요."로 작성한다.
8. 가격 외의 숫자는 사용하지 않는다.



> 💡 노트북에서는 상품 정보와 규칙을 코드에 직접 적었다. 실제 서비스에서는 **코드 밖**에서 관리한다 → 상품이 늘거나 규칙이 바뀌어도 코드를 다시 배포하지 않기 위해서다.

<details>
<summary>실제 서비스에서 관리하는 곳 (펼쳐 보기)</summary>

| 대상 | 관리하는 곳 | 바꾸는 사람 |
| --- | --- | --- |
| 상품 정보 | 상품 DB (쇼핑몰 관리자 화면에서 등록) → 서비스는 상품 ID로 조회하거나 요청으로 받는다 | 상품 담당자(MD) |
| 상품 소개 규칙 | 규칙 설정 파일(YAML·JSON)·DB 테이블, 또는 프롬프트 관리 도구 (예: LangSmith Prompt Hub, Langfuse) | 마케팅·법무(광고 심의)가 정하고 개발자가 반영 |
| 검사기 | 코드 (Git 저장소) — 규칙 번호마다 검사 한 줄 | 개발자 |

- 규칙 문장과 검사기는 **함께** 바뀌어야 한다 → 번호(R1~R8)로 짝을 맞추고, 규칙을 고치면 검사기 시험을 다시 돌린다 (M2 Day 8 CI의 `test_checker.py`).
- 규칙에는 버전을 붙인다 → "어느 규칙 버전으로 쓰고 검사했나"를 기록에 남겨야 나중에 결과를 설명할 수 있다.
- 상품 정보는 요청에 실려 온다 → M2 Day 6 서비스는 요청 본문으로 `name`·`price`를 받는다.
- 규칙을 "코드로 검사할 수 있는 문장"으로 쓰는 일은 M3 Day 13 정책 문서에서 다시 한다.
</details>

### 1-2. 검사기 — 판정은 코드로 직접 한다

소개문이 규칙을 지켰는지 검사하는 함수를 만든다. 위반 목록을 반환한다.

> 💡 "검사를 왜 LLM에 안 맡기나?"는 2-3에서 직접 실험한다. 먼저 결정적(항상 같은 판정) 검사기를 만들어 둔다.

In [3]:
# 검사기 에이전트
# text: 상품소개문
# product: 상품정보 - 상품소개문 생성할 때 사용한 정보
def check_rules(text, product):
    """상품 소개문이 작성 규칙을 지켰는지 검사한다. 위반이 없으면 []."""
    violations = [] # 위반내역
    name = product["name"]
    price = product["price"]
    text = text.strip()
    # 가격 안의 쉼표·숫자는 허용해야 하므로, R6·R8은 가격을 지운 문장으로 검사한다
    text_without_price = text.replace(price, "")

    if text.count(".") != 3:
        violations.append(f"R1 문장 수 위반: 마침표가 {text.count('.')}개입니다.")
    if name not in text:
        violations.append(f'R2 제품명 누락: "{name}"이 없습니다.')
    if price not in text:
        violations.append(f'R3 가격 누락: "{price}"이 없습니다.')
    used = [word for word in ["최고", "완벽", "혁신", "프리미엄", "특별"] if word in text]
    if used:
        violations.append(f"R4 금지 단어 사용: {used}")
    marks = [m for m in ["!", "?"] if m in text]
    if marks:
        violations.append(f"R5 금지 문장부호 사용: {marks}")
    if "," in text_without_price:
        violations.append("R6 가격 외의 쉼표 사용")
    if not text.endswith("지금 만나보세요."):
        violations.append('R7 마지막 문장은 "지금 만나보세요."여야 합니다.')
    if re.search(r"\d", text_without_price):
        violations.append("R8 가격 외의 숫자 사용")
    return violations

In [4]:
# 일부러 가격을 빠뜨린 예문 → 검사기가 잡아내는지 확인
wrong_text = "숲속 캠핑 의자는 가벼워서 휴대하기 편합니다. 접이식 디자인과 컵홀더를 갖추고 있습니다. 지금 만나보세요."
print("틀린 예문:", check_rules(wrong_text, product))

# 규칙을 모두 지킨 예문 → 빈 리스트가 나와야 한다
correct_text = "숲속 캠핑 의자는 가벼워서 휴대하기 편합니다. 접이식 디자인과 컵홀더를 갖췄으며 가격은 29,900원입니다. 지금 만나보세요."
print("맞는 예문:", check_rules(correct_text, product))

틀린 예문: ['R3 가격 누락: "29,900원"이 없습니다.']
맞는 예문: []


### 1-3. 단일 에이전트 — 규칙 8개 일괄 지시

에이전트 하나에게 상품 정보 + 규칙 8개를 한 번에 지시하고, 결과를 검사기로 채점한다.

In [5]:
def get_token_usage(response):
    """LLM 응답의 전체 토큰 수. 없으면 0."""
    usage = response.usage_metadata
    if not usage:
        return 0
    return usage.get("total_tokens", 0)


def build_task_prompt(product):
    """상품 정보 + 규칙을 하나의 지시문으로 만든다."""
    rules = PRODUCT_INTRO_RULES.format(name=product["name"], price=product["price"])
    features = "\n".join(f"- {f}" for f in product["features"])
    return f"""
당신은 온라인 쇼핑몰의 상품 소개문 작성자입니다.
다음 상품 정보를 사용하여 상품 소개문을 작성하세요.

[상품 정보]
제품명: {product["name"]}
가격: {product["price"]}
특징:
{features}
{rules}
상품 소개문만 출력하세요. 규칙 설명이나 인사말은 출력하지 마세요.
""".strip()


def run_single(product):
    """단일 에이전트: 규칙 8개를 한 번에 지시한다. 반환: (text, tokens, sec)"""
    start = time.perf_counter()
    # TODO: 초안을 쓰는 모델로 build_task_prompt(product)를 호출해 결과를 response에 담으세요
    response = llm_writer.invoke(build_task_prompt(product))

    sec = time.perf_counter() - start
    return response.content.strip(), get_token_usage(response), sec

In [6]:
single_text, single_tokens, single_sec = run_single(product)
single_violations = check_rules(single_text, product)

print("[단일 에이전트 결과]")
print(single_text)
print("\n[규칙 위반]", single_violations)
print("토큰:", single_tokens, "/ 시간:", round(single_sec, 2), "초")

[단일 에이전트 결과]
숲속 캠핑 의자는 가벼워서 휴대하기 편안하며, 접이식 디자인으로 휴대성이 뛰어납니다. 또한 컵홀더가 포함되어 있어 야외 활동에 적합한 아이템입니다. 지금 만나보세요.

[규칙 위반] ['R3 가격 누락: "29,900원"이 없습니다.', 'R6 가격 외의 쉼표 사용']
토큰: 319 / 시간: 2.09 초


**관찰 포인트**: 어떤 규칙이 깨졌는지 확인한다. 여러 번 실행해 매번 같은 규칙이 깨지는지도 본다.

### 1-4. 협업 파이프라인 — 작성자·수정자 분리

```text
[작성자 Agent] → 초안 → [검사기(코드)] → 위반 목록 → [수정자 Agent] → 최종본
```

| 역할 | 책임 | 하지 않는 것 |
| --- | --- | --- |
| 작성자 | 초안 작성 | 검토 |
| 검사기 | 위반을 정확히 찾기 (코드라서 환각 없음) | 수정 |
| 수정자 | 지적받은 것만 고치기 | 새로 쓰기 |

> ⚠️ 수정자 프롬프트의 핵심: **"새 문장을 추가하지 말고 기존 문장 안에 녹여 넣으세요"**. 이 지시가 없으면 빠진 가격을 4번째 문장으로 붙여 문장 수 규칙(R1)을 깨뜨린다.

In [12]:
def writer_agent(product):
    """작성자 역할: 초안을 작성한다."""
    # TODO: 초안을 쓰는 모델(llm_writer)로 build_task_prompt(product)를 호출해 결과를 response에 담으세요
    response = llm_writer.invoke(build_task_prompt(product))
    return response.content.strip(), get_token_usage(response)

In [22]:
# [수정 원칙]
# - 위반한 부분을 고치세요.
# - 이미 잘 지킨 부분은 가능한 한 유지하세요.
# - ____ (TODO: 빠진 내용을 어떻게 넣으라고 해야 문장 수 규칙이 안 깨질까요?)
# - 상품 소개문만 출력하세요. 수정 설명이나 인사말은 출력하지 마세요.

def reviser_agent(draft, violations, product):
    """수정자 역할: 검사기가 찾은 위반만 고친다."""
    
    rules = PRODUCT_INTRO_RULES.format(name=product["name"], price=product["price"])
    violation_text = "\n".join(f"- {v}" for v in violations)
    
    prompt = f"""
당신은 상품 소개문 수정자입니다.
작성자가 만든 초안에서 검사기가 규칙 위반을 발견했습니다. 검사기가 알려 준 문제를 수정하세요.

[초안]
{draft}

[검사기가 발견한 위반]
{violation_text}
{rules}
[수정 원칙]
- 위반한 부분을 고치세요.
- 이미 잘 지킨 부분은 가능한 한 유지하세요.
- 새 문장을 추가하지 말고 기존 문장 안에 자연스럽게 넣으세요. (반드시 문장 갯수는 3문장을 유지)
- 상품 소개문만 출력하세요. 수정 설명이나 인사말은 출력하지 마세요.
""".strip()

    # TODO: 수정 지시를 흔들림 없이 따라야 하는 모델(llm_fixer)로 prompt를 호출해 결과를 response에 담으세요
    response = llm_fixer.invoke(build_task_prompt(product))
    return response.content.strip(), get_token_usage(response)

In [ ]:
# 멀티에이전트 파이프라인
def run_multi(product):
    """협업 파이프라인: 작성자 → 검사기 → 수정자(1회) → 재검사.

    반환: draft, draft_violations, final, final_violations, tokens, sec
    """
    start = time.perf_counter()
    draft, tokens = writer_agent(product)
    # TODO: 초안을 검사기로 채점해 draft_violations에 담으세요
    draft_violations = check_rules(draft, product)
    
    # TODO: 위반이 있을 때만 수정자(reviser_agent)를 불러 final을 만들고, 수정자 토큰을 tokens에 더하세요
    #       위반이 없으면 수정자를 부르지 않고 final = draft (→ 토큰을 아낀다)
    if draft_violations:
        final, fixer_tokens = reviser_agent(draft, draft_violations, product)
        tokens += fixer_tokens
    else :
        final = draft

    # 응답의 최종 확인
    final_violations = check_rules(final, product)
    return draft, draft_violations, final, final_violations, tokens, time.perf_counter() - start

In [ ]:
# 멀티에이전트 실행
draft, draft_violations, final, final_violations, multi_tokens, multi_sec = run_multi(product)
# TODO: 초안·초안 위반(draft_violations)·최종본·최종 위반(final_violations)을 차례로 출력하세요
#      (1-3의 단일 에이전트 결과 출력 형식을 참고)

print("[1단계:작성자 초안]", draft)
print("\n[2단계:작성자 초안 위반]", draft_violations)
print("\n[3단계:최종본]",final)
print("\n[4단계:최종 위반]", final_violations)
print("토큰:", multi_tokens, "/ 시간:", round(multi_sec, 2), "초")

[1단계:작성자 초안] 숲속 캠핑 의자는 가벼워서 휴대하기 편리합니다. 접이식 디자인과 컵홀더가 포함되어 있어 야외 활동에 적합합니다. 지금 만나보세요.

[2단계:작성자 초안 위반] ['R3 가격 누락: "29,900원"이 없습니다.']

[3단계:최종본] 숲속 캠핑 의자는 가벼워서 휴대하기 편리하며, 접이식 디자인으로 공간을 절약할 수 있습니다. 컵홀더가 포함되어 있어 야외 활동 시 더욱 유용하며, 가격은 29,900원입니다. 지금 만나보세요.

[4단계:최종 위반] ['R6 가격 외의 쉼표 사용']
토큰: 632 / 시간: 2.21 초


### 1-5. 단일 vs 협업 비교

비교 결과를 표로 확인한다.

In [27]:
pd.DataFrame([
    {"구조": "단일 에이전트", "위반 수": len(single_violations), "토큰": single_tokens, "시간(초)": round(single_sec, 2)},
    {"구조": "협업 파이프라인", "위반 수": len(final_violations), "토큰": multi_tokens, "시간(초)": round(multi_sec, 2)},
])

,구조,위반 수,토큰,시간(초)
0,단일 에이전트,2,319,2.09
1,협업 파이프라인,1,632,2.21


**관찰 포인트**: 1회 실행 결과만으로 "협업이 낫다"고 말할 수 있는가? → Part 2에서 반복 실행으로 확인한다.

## Part 2. 바꿔 보기 — 협업은 언제, 왜 유리한가

**전체 그림 2/3**: Part 1의 `run_single` · `run_multi`를 조건만 바꿔 다시 돌린다 → 협업이 유리한 조건과 대가를 찾는다.

조건을 하나씩 바꾸며 "멀티가 언제 유리한지"를 숫자로 확인한다.

### 2-1. 규칙 수 조절 — 규칙이 적으면 단일도 잘한다

규칙을 3개(문장 수·제품명·가격)로 줄여 단일 에이전트를 실행한다.

<details>
<summary>쉬운 버전 검사기가 따로 필요한 이유</summary>

- 규칙 5(느낌표 금지)가 없으면 "만나보세요!"처럼 끝날 수 있다.
- 마침표만 세면 이런 문장이 억울하게 위반 판정을 받는다 → 문장 끝 부호(`.` `!` `?` `…`)를 모두 센다.
</details>

In [31]:
# 상품 소개 규칙 (쉬운 버전: 3개)
PRODUCT_INTRO_RULES_EASY = """
[작성 규칙]
1. 정확히 3문장으로 쓴다.
2. 제품명 "{name}"을 그대로 포함한다.
3. 가격 "{price}"를 그대로 포함한다.
"""


def check_rules_easy(text, product):
    """쉬운 버전 검사기: 규칙 3개만 검사한다."""
    violations = []
    # TODO: . ! ? … 가 1개 이상 이어진 묶음을 re.findall로 찾아 sentence_marks에 담으세요 (묶음 수 = 문장 수)
    sentence_marks = re.findall(r'[.!?…]+', text)
    if len(sentence_marks) != 3:
        violations.append(f"R1 문장 수 위반: 문장 끝 부호가 {len(sentence_marks)}개입니다.")
    if product["name"] not in text:
        violations.append("R2 제품명 누락")
    if product["price"] not in text:
        violations.append("R3 가격 누락")
    return violations


def run_single_easy(product):
    """규칙 3개짜리 쉬운 과제로 단일 에이전트를 실행한다."""
    
    rules = PRODUCT_INTRO_RULES_EASY.format(name=product["name"], price=product["price"])
    
    prompt = f"""
당신은 온라인 쇼핑몰의 상품 소개문 작성자입니다.
제품명: {product["name"]} / 가격: {product["price"]}
{rules}
상품 소개문만 출력하세요.
""".strip()

    # TODO: llm_writer로 prompt를 호출하고, 아래 return ""를 응답 내용(content.strip())을 반환하도록 바꾸세요
    
    return llm_writer.invoke(prompt).content.strip()


for i in range(3):
    easy_text = run_single_easy(product)
    print(f"[{i + 1}회] 위반 {check_rules_easy(easy_text, product)}")
    print("   ", easy_text)

[1회] 위반 []
    자연 속에서 편안함을 만끽할 수 있는 숲속 캠핑 의자는 휴식과 여유를 제공합니다. 견고한 소재로 제작되어 내구성이 뛰어나며, 가벼운 무게로 휴대가 간편합니다. 지금 바로 숲속 캠핑 의자를 29,900원에 만나보세요!
[2회] 위반 []
    자연의 품에서 편안함을 느낄 수 있는 숲속 캠핑 의자를 소개합니다. 이 의자는 휴대가 간편하고 내구성이 뛰어나, 캠핑뿐만 아니라 다양한 야외 활동에 적합합니다. 합리적인 가격 29,900원으로 당신의 캠핑 경험을 더욱 특별하게 만들어 보세요!
[3회] 위반 []
    편안함과 휴식을 동시에 제공하는 숲속 캠핑 의자입니다. 야외에서도 안락한 좌석을 경험할 수 있도록 설계되었으며, 가벼운 무게로 휴대가 간편합니다. 가격은 29,900원으로, 캠핑과 피크닉에 최적인 아이템입니다.


**관찰 포인트**: 규칙이 3개일 때 단일 에이전트의 위반 수는 1-3과 비교해 어떤가?

### 2-2. 자기 검토 — 역할 분리 없는 검토의 한계

에이전트를 나누지 않고, 한 에이전트에게 "작성 후 스스로 검토까지" 하는 구조로 작성한다.


In [34]:
def run_self_review(product):
    """단일 에이전트에 '스스로 검토'까지 시킨다. (역할 분리 없음)"""
    # TODO: build_task_prompt(product) 뒤에 "작성 후 스스로 규칙을 검토해 고친 최종본만 출력하라"는 지시를 붙여 prompt를 만드세요
    prompt = build_task_prompt(product) + "작성 후 스스로 규칙을 검토해 고친 최종본만 출력하라"
    response = llm_writer.invoke(prompt)
    return response.content.strip()


for i in range(3):
    self_text = run_self_review(product)
    print(f"[{i + 1}회] 위반 {check_rules(self_text, product)}")

[1회] 위반 ['R3 가격 누락: "29,900원"이 없습니다.']
[2회] 위반 ['R3 가격 누락: "29,900원"이 없습니다.', 'R6 가격 외의 쉼표 사용']
[3회] 위반 ['R3 가격 누락: "29,900원"이 없습니다.', 'R6 가격 외의 쉼표 사용']


**관찰 포인트**: "스스로 검토하라"는 한 줄로 위반이 사라졌는가?

### 2-3. LLM 단독 검토자 — 검토 환각 관찰

같은 초안을 LLM 검토자와 검사기에게 각각 판정시킨다.

In [37]:
def llm_reviewer_check(draft, product):
    """LLM에게 규칙 점검을 맡긴다. (검사기 없이)"""
    rules = PRODUCT_INTRO_RULES.format(name=product["name"], price=product["price"])
    
    # TODO: 아래 prompt를 완성하세요 — 검토자 역할 + [초안] draft + 규칙(rules)을 넣고,
    #      규칙 1~8 각각 지켰는지 판정해 위반한 규칙 번호와 이유를 목록으로 출력하라고 지시
    prompt = f"""
당신은 상품 소개문 검토자이다.
규칙 1~8 각각 지켰는지 판정해 위반한 규칙 번호와 이유를 목록으로 출력하라.

[초안]
{draft}

[규칙]
{rules}
""".strip()

    return llm_fixer.invoke(prompt).content.strip()


draft, _ = writer_agent(product)
print("[초안]\n" + draft)
print("\n[LLM 검토자의 판정]\n" + llm_reviewer_check(draft, product))
# TODO: 같은 초안을 검사기로 채점해 "[검사기의 판정]"으로 출력하고 LLM 판정과 비교하세요
print("\n[검사기의 판정]\n", check_rules(draft, product))

[초안]
숲속 캠핑 의자는 가벼워서 휴대하기 편리하며, 접이식 디자인으로 편리함을 더했습니다. 컵홀더가 포함되어 있어 야외 활동에 적합한 이 의자는 29,900원에 만나보실 수 있습니다. 지금 만나보세요.

[LLM 검토자의 판정]
위반한 규칙 번호와 이유는 다음과 같습니다.

1. **위반 규칙 1**: 문장이 3개가 아닌 4개로 작성되었습니다. (문장 수 초과)
2. **위반 규칙 6**: 쉼표가 가격 표기 외에 사용되지 않아야 하지만, 문장 내에서 쉼표가 사용되지 않았습니다. (문장 내 쉼표 사용)
3. **위반 규칙 8**: 가격 외의 숫자는 사용하지 않아야 하지만, "29,900원" 외에 다른 숫자가 사용되지 않았습니다. (이 규칙은 위반되지 않음)

결론적으로, 위반된 규칙은 1번과 6번입니다.

[검사기의 판정]
 ['R6 가격 외의 쉼표 사용']


**관찰 포인트**: LLM 검토자의 판정과 검사기의 판정이 어긋나는 규칙이 있는가? 있다면 누가 맞았는가?

### 2-4. 반복 실행 통계 — 1회 운이 아닌 경향

temperature가 0.9라 결과가 매번 다르다 → 각 5회씩 실행해 평균으로 비교한다.

> ⚠️ 이 셀은 LLM을 최대 15번 호출한다 (단일 5회 + 협업 5회×최대 2). 1~2분 걸린다.

In [40]:
N = 5
records = []
for mode in ["단일", "협업"]:
    for i in range(N):
        if mode == "단일":
            # TODO: 단일 에이전트 함수로 text, tokens, sec를 받고, 검사기로 채점해 violations에 담으세요
            text, tokens, sec = run_single(product)
            violations = check_rules(text, product)
        else:
            # TODO: 협업 파이프라인 함수(반환값 6개)에서 최종본 text, 최종 위반 violations, tokens, sec를 받으세요
            draft, draft_violations, final, violations, tokens, sec = run_multi(product)

        records.append({"구조": mode, "위반 수": len(violations), "토큰": tokens, "시간(초)": round(sec, 2)})
        print(f"{mode} {i + 1}회: 위반 {violations}")

단일 1회: 위반 ['R3 가격 누락: "29,900원"이 없습니다.']
단일 2회: 위반 ['R3 가격 누락: "29,900원"이 없습니다.']
단일 3회: 위반 ['R3 가격 누락: "29,900원"이 없습니다.', 'R6 가격 외의 쉼표 사용']
단일 4회: 위반 ['R6 가격 외의 쉼표 사용']
단일 5회: 위반 ['R3 가격 누락: "29,900원"이 없습니다.', 'R6 가격 외의 쉼표 사용']
협업 1회: 위반 ['R6 가격 외의 쉼표 사용']
협업 2회: 위반 ['R6 가격 외의 쉼표 사용']
협업 3회: 위반 ['R6 가격 외의 쉼표 사용']
협업 4회: 위반 ['R6 가격 외의 쉼표 사용']
협업 5회: 위반 ['R6 가격 외의 쉼표 사용']


### 2-5. 품질·비용·속도 트레이드오프

평균으로 정리하면 협업의 효과와 대가가 함께 보인다.

In [41]:
records_df = pd.DataFrame(records)
records_df.groupby("구조").mean().round(2)

,위반 수,토큰,시간(초)
구조,,,
단일,1.4,317.8,1.06
협업,1.0,649.2,2.19


**관찰 포인트**: 협업의 위반 수는 줄었는가? 토큰은 단일의 몇 배인가? 2-4 출력에서 협업이 **못 고친 규칙**은 무엇인가?

## Part 2-확장. 수정 루프 — 한 번 고쳐서 안 되면

지금까지 수정자는 **한 번만** 고쳤다. 
고친 결과를 다시 검사해 위반이 남으면 또 고치게 한다.

```text
작성자 → 검사기 → 위반 있음? ─ 아니오 → 종료
                      └ 예 → 수정자 → 검사기 → (최대 N번 반복)
```

> 💡 이 반복 구조가 Day 5의 **피드백 루프**다. 오늘은 반복 횟수 상한(`max_revision`)만 두고 효과와 대가를 먼저 본다.

### 2-6. 수정 루프 구현

In [42]:
def run_multi_loop(product, max_revision=3):
    """작성자 → (검사기 → 수정자)를 위반이 없어질 때까지, 최대 max_revision번 반복한다.

    반환: final, final_violations, revision_log, tokens, sec
    """
    start = time.perf_counter()
    
    # TODO: 작성자(writer_agent)로 첫 초안을 만들어 current·tokens에 담고, 검사기로 채점해 violations에 담으세요
    current, tokens = writer_agent(product) # 작성자 에이전트
    violations = check_rules(current, product) # 검토기
    
    revision_log = [{"회차": 0, "위반": violations, "문장": current}]

    for n in range(1, max_revision + 1):
        # TODO: 멈춤 조건 — 위반이 없으면 반복을 끝내세요
        if not violations:
            break

        # 수정자 에이전트 호출
        current, fixer_tokens = reviser_agent(current, violations, product)
        tokens += fixer_tokens
        
        # TODO: 고친 결과를 반드시 다시 검사해 violations를 갱신하세요
        violations = check_rules(current, product) # 검토기
        revision_log.append({"회차": n, "위반": violations, "문장": current})

    # 최종 소개문, 위반사항([], 최대횟수 수행후 위반사항)
    return current, violations, revision_log, tokens, time.perf_counter() - start

In [48]:
# TODO: 수정 루프 함수(run_multi_loop)를 product로 실행해 반환값 5개를 받으세요 (= None을 바꾸세요)
final, final_violations, revision_log, loop_tokens, loop_sec = run_multi_loop(product, max_revision=3)

for log in revision_log:
    print(f"[{log['회차']}차] 위반 {log['위반']}")  # 0차 = 작성자 초안
    print("   ", log["문장"])

print("수정 횟수:", len(revision_log) - 1, "/ 최종 위반:", final_violations, "/ 토큰:", loop_tokens)

[0차] 위반 ['R6 가격 외의 쉼표 사용']
    가벼워서 휴대하기 편한 숲속 캠핑 의자는 접이식 디자인으로 야외 활동에 적합합니다. 컵홀더가 포함되어 있어 편리함을 더해 줍니다. 가격은 29,900원이며, 지금 만나보세요.
[1차] 위반 ['R6 가격 외의 쉼표 사용']
    숲속 캠핑 의자는 가벼워서 휴대하기 편리하며, 접이식 디자인으로 손쉽게 보관할 수 있습니다. 컵홀더가 포함되어 있어 야외 활동 시 더욱 유용하며, 가격은 29,900원입니다. 지금 만나보세요.
[2차] 위반 ['R6 가격 외의 쉼표 사용']
    숲속 캠핑 의자는 가벼워서 휴대하기 편리하며, 접이식 디자인으로 손쉽게 보관할 수 있습니다. 컵홀더가 포함되어 있어 야외 활동 시 더욱 유용하며, 가격은 29,900원입니다. 지금 만나보세요.
[3차] 위반 ['R6 가격 외의 쉼표 사용']
    숲속 캠핑 의자는 가벼워서 휴대하기 편리하며, 접이식 디자인으로 손쉽게 보관할 수 있습니다. 컵홀더가 포함되어 있어 야외 활동 시 더욱 유용하며, 가격은 29,900원입니다. 지금 만나보세요.
수정 횟수: 3 / 최종 위반: ['R6 가격 외의 쉼표 사용'] / 토큰: 1302


### 2-7. 1회 수정 vs 수정 루프 — 반복 통계

2-4의 협업(1회 수정) 기록에 수정 루프 5회를 더해 비교한다.

> ⚠️ 이 셀은 LLM을 최대 20번 호출한다 (5회 × 최대 4). 1~2분 걸린다.

In [49]:
N = 10

In [51]:
for i in range(N):
    # TODO: 수정 루프 함수(run_multi_loop)를 product로 실행해 반환값 5개를 받으세요 (= None을 바꾸세요)
    _, violations, log, tokens, sec = run_multi_loop(product, max_revision=3)
    
    records.append({"구조": "루프", "위반 수": len(violations), "토큰": tokens, "시간(초)": round(sec, 2)})
    print(f"루프 {i + 1}회: 수정 {len(log) - 1}번 / 위반 {violations}")

# 2-4의 records 목록에 루프 기록을 이어 붙였으니, 세 방식을 한 표로 비교할 수 있다
pd.DataFrame(records).groupby("구조").mean().round(2)

루프 1회: 수정 3번 / 위반 ['R6 가격 외의 쉼표 사용']
루프 2회: 수정 3번 / 위반 ['R6 가격 외의 쉼표 사용']
루프 3회: 수정 3번 / 위반 ['R6 가격 외의 쉼표 사용']
루프 4회: 수정 3번 / 위반 ['R6 가격 외의 쉼표 사용']
루프 5회: 수정 3번 / 위반 ['R6 가격 외의 쉼표 사용']
루프 6회: 수정 3번 / 위반 ['R6 가격 외의 쉼표 사용']
루프 7회: 수정 3번 / 위반 ['R6 가격 외의 쉼표 사용']
루프 8회: 수정 3번 / 위반 ['R6 가격 외의 쉼표 사용']
루프 9회: 수정 3번 / 위반 ['R6 가격 외의 쉼표 사용']
루프 10회: 수정 3번 / 위반 ['R6 가격 외의 쉼표 사용']


,위반 수,토큰,시간(초)
구조,,,
단일,1.4,317.80,1.06
루프,1.0,1297.64,4.46
협업,1.0,649.20,2.19


**관찰 포인트**

- 루프는 1회 수정보다 위반을 더 줄였는가? 대가(토큰·시간)는?
- 3번을 고쳐도 위반이 남은 회차가 있다면, 수정 이력에서 **무엇을 고치다 무엇이 깨졌는지** 찾아본다.

## Part 3. 미니 프로젝트 — "단일 vs 협업 vs 루프" 비교 리포트

**전체 그림 3/3**: Part 2까지 만든 방식 3종(단일 · 협업 · 루프)을 상품 3종에 돌린다 → 숫자로 결론을 낸다.

성격이 다른 상품 3종으로 실험을 넓혀, 품질·비용·속도 비교 리포트를 완성한다.

| 방식 | 구성 |
| --- | --- |
| 단일 | 에이전트 1개, 규칙 8개 일괄 지시 |
| 협업 | 작성자 → 검사기 → 수정자(1회) |
| 루프 | 작성자 → (검사기 → 수정자) 최대 3회 |

### 3-1. 실험 대상 — 상품 3종

가구·식품·전자기기를 고른다. 규칙과 검사기는 그대로 쓴다.

In [ ]:
PRODUCTS = [
    product,  # 숲속 캠핑 의자
    # TODO: 식품·전자기기 상품 2개를 product와 같은 형식(name · price · features 4개)으로 추가하세요
    #      가격은 "12,900원"처럼 쉼표 표기, 특징에는 숫자를 넣지 마세요 (R6·R8 때문에)
    
]
for p in PRODUCTS:
    print(p["name"], "-", p["price"])

### 3-2. 실험 실행 — 상품 3종 × 방식 3종 × 3회

> ⚠️ 이 셀은 LLM을 60번 안팎 호출한다. 3~5분 걸린다. 실행 중에 3-3의 리포트 틀을 먼저 읽어 둔다.

In [ ]:
REPEAT = 3
rows = []
for prod in PRODUCTS:
    for mode in ["단일", "협업", "루프"]:
        for i in range(REPEAT):
            if mode == "단일":
                # TODO: 단일 에이전트 함수(run_single)를 prod로 실행해 text, tokens, sec를 받으세요 (= None을 바꾸세요)
                text, tokens, sec = None
                violations = check_rules(text, prod)
            elif mode == "협업":
                # TODO: 협업 파이프라인 함수(run_multi)를 prod로 실행해 반환값 6개 중 최종본·최종 위반·tokens·sec를 받으세요 (= None을 바꾸세요)
                _, _, text, violations, tokens, sec = None
            else:
                # TODO: 수정 루프(max_revision=3, 반환값 5개)에서 최종본 text, 최종 위반 violations, tokens, sec를 받으세요
                
            # TODO: rows에 {"상품": 상품명, "구조": mode, "위반 수", "토큰", "시간(초)"} dict를 추가하세요 (rows.append("")를 바꾸세요)
            rows.append("")
            
            # TODO: 끝까지 남은 위반만 출력 → 어떤 규칙이 잘 안 고쳐지는지 본다
            
    print(f"{prod['name']}: 완료")

result_df = pd.DataFrame(rows)

### 3-3. 집계 — 방식별·상품별 평균

In [ ]:
result_df.groupby("구조")[["위반 수", "토큰", "시간(초)"]].mean().round(2)

In [ ]:
# 상품별로 보면 방식의 효과가 고르게 나타나는지 확인한다
result_df.groupby(["상품", "구조"]).mean().round(2)

### 3-4. 비교 리포트 — 결론 3줄

집계 표를 근거로 3줄을 쓴다. **숫자를 반드시 넣는다.**

1. **품질**: 방식별 평균 위반 수는 어떻게 달라졌는가?
2. **비용·속도**: 협업·루프의 토큰·시간은 단일의 몇 배인가?
3. **판단 기준**: 어떤 조건이면 단일로 충분하고, 어떤 조건이면 협업·루프를 쓰겠는가?

> (여기에 자신의 결론을 작성합니다)

## Part 3-확장. 다른 도메인에 적용 — 파이프라인 일반화

지금 파이프라인은 **상품 소개문 전용**이다 (`build_task_prompt`·`check_rules`가 안에 고정). 지시문과 검사기를 **인자로 받게** 바꾸면 규칙이 있는 다른 작성 작업에도 쓸 수 있다.

> 💡 역할(작성·검사·수정)은 그대로, 내용만 갈아 끼운다. Day 2에서 이 "역할과 주고받는 형식"을 체계적으로 설계한다.

### 3-5. 범용 파이프라인

<details>
<summary>함수를 인자로 넘긴다는 것 (펼쳐 보기)</summary>

- 파이썬에서는 함수도 값처럼 넘길 수 있다. `run_pipeline(지시문, check_notice)`처럼 **괄호 없이** 이름만 넘긴다.
- 받은 쪽은 `checker(text)`처럼 괄호를 붙여 호출한다 → 어떤 검사기를 넘기느냐에 따라 검사 내용이 바뀐다.
</details>

In [56]:
def run_pipeline(task_prompt, checker, max_revision=2):
    """범용 협업 파이프라인: 작성자 → (검사기 → 수정자) 반복. 반환: (최종 글, 위반 목록)"""
    text = llm_writer.invoke(task_prompt).content.strip()

    # TODO: 인자로 받은 검사기(checker)로 text를 채점해 violations에 담으세요
    violations = checker(text)

    for n in range(max_revision):
        if not violations:
            break

        violation_text = "\n".join(f"- {v}" for v in violations)
        
        # TODO: fix_prompt를 완성하세요 — 원래 과제(task_prompt), 현재 글(text), 위반 목록(violation_text)을 넣고,
        #      위반만 고치고 새 문장을 추가하지 말고 기존 문장 안에 녹여 넣으며 수정한 글만 출력하라고 지시
        fix_prompt = f"""
다음 글이 규칙을 위반했습니다. 위반만 고치고 나머지 내용은 유지하세요. 고친 글만 출력하세요.

[원래 지시]
{task_prompt}

[글]
{text}

[위반]
{violation_text}

""".strip()
        text = llm_fixer.invoke(fix_prompt).content.strip()
        # TODO: 고친 text를 다시 채점해 violations를 갱신하세요
        violations = checker(text)

    return text, violations

# 범용에이전트 테스트
# 기존 상품 소개문 과제로 먼저 확인한다 → 검사기가 text 하나만 받도록 감싼다
def check_product(text):
    return check_rules(text, product)

# TODO: run_pipeline에 상품 소개 지시문 build_task_prompt(product)와 검사기 check_product를 넘겨 실행하세요
#      (함수는 괄호 없이 이름만 넘긴다, = "", ""를 바꾸세요)
final, violations = run_pipeline(build_task_prompt(product), check_product)
print("최종 위반:", violations)
print(final)

최종 위반: ['R1 문장 수 위반: 마침표가 4개입니다.']
숲속 캠핑 의자는 가벼워서 휴대하기 편리하며 접이식 디자인으로 간편하게 보관할 수 있습니다. 또한 컵홀더가 포함되어 있어 야외 활동 시 더욱 유용합니다. 가격은 29,900원입니다. 지금 만나보세요.


### 3-6. 새 도메인 — 매장 휴무 안내 문자

규칙이 있는 다른 작성 작업을 골랐다. 검사기만 새로 만들고, 파이프라인은 3-5를 그대로 쓴다.

| 규칙 | 내용 | 코드 판정 |
| --- | --- | --- |
| N1 | 정확히 2문장 | 마침표 2개 |
| N2 | 매장명 포함 | 포함 여부 |
| N3 | 휴무일 "10월 9일"·재개일 "10월 10일" 그대로 포함 | 포함 여부 |
| N4 | 공백 포함 40자 이내 | `len()` |
| N5 | "양해" 표현 포함 | 포함 여부 |
| N6 | 쉼표·느낌표·물음표 금지 | 포함 여부 |

In [57]:
store = {"name": "초록 베이커리", "holiday": "10월 9일", "reopen": "10월 10일", "reason": "한글날"}

# TODO: NOTICE_PROMPT를 완성하세요 — store의 매장명·휴무일·재개일·사유와 규칙 N1~N6을 넣고, 안내 문자만 출력하라고 지시
NOTICE_PROMPT = f"""
당신은 동네 가게의 고객 안내 문자 작성자입니다.
[정보] 매장명: {store["name"]} / 휴무일: {store["holiday"]} / 영업 재개일: {store["reopen"]} / 사유: {store["reason"]}
[작성 규칙]
1. 정확히 2문장으로 쓴다. 마침표는 정확히 2개 사용한다.
2. 매장명 "{store["name"]}"을 그대로 포함한다.
3. 휴무일 "{store["holiday"]}"과 영업 재개일 "{store["reopen"]}"을 그대로 포함한다.
4. 공백 포함 40자 이내로 쓴다.
5. "양해"라는 단어를 포함한다.
6. 쉼표, 느낌표, 물음표를 사용하지 않는다.
문자 내용만 출력하세요.
""".strip()


def check_notice(text):
    """안내 문자 검사기. 위반이 없으면 []."""
    text = text.strip()
    violations = []
    if text.count(".") != 2:
        violations.append(f"N1 문장 수 위반: 마침표 {text.count('.')}개")
    if store["name"] not in text:
        violations.append("N2 매장명 누락")
    if store["holiday"] not in text or store["reopen"] not in text:
        violations.append("N3 날짜 누락")
    # TODO: N4 — 공백 포함 40자를 넘으면 violations에 f"N4 길이 초과: {len(text)}자"를 추가하세요
    if len(text) > 40:
        violations.append(f"N4 길이 초과: {len(text)}자")
    if "양해" not in text:
        violations.append("N5 양해 표현 누락")
    marks = [m for m in [",", "!", "?"] if m in text]
    if marks:
        violations.append(f"N6 금지 문장부호: {marks}")
    return violations

In [61]:
# 단일 에이전트 vs 범용 파이프라인, 각 3회
for i in range(3):
    # TODO: 단일 에이전트: llm_writer로 NOTICE_PROMPT를 한 번 호출해 결과 문장을 text에 담으세요
    text = llm_writer.invoke(NOTICE_PROMPT).content.strip()
    print(f"[단일 {i + 1}회] {len(text)}자 / 위반 {check_notice(text)}")
    print("   ", text)

    # TODO: 범용 파이프라인: run_pipeline에 NOTICE_PROMPT와 검사기 check_notice를 넘겨 final, violations를 받으세요
    final, violations = run_pipeline(NOTICE_PROMPT, check_notice)
    print(f"[파이프라인 {i + 1}회] {len(final)}자 / 위반 {violations}")
    print("   ", final)

[단일 1회] 48자 / 위반 ['N1 문장 수 위반: 마침표 1개', 'N4 길이 초과: 48자']
    초록 베이커리는 10월 9일 휴무입니다 양해 부탁드리며 10월 10일 영업 재개합니다.
[파이프라인 1회] 48자 / 위반 ['N4 길이 초과: 48자']
    초록 베이커리는 10월 9일에 휴무입니다. 양해 부탁드리며 10월 10일에 재개합니다.
[단일 2회] 50자 / 위반 ['N4 길이 초과: 50자']
    초록 베이커리는 10월 9일 휴무입니다. 10월 10일에 영업 재개하니 양해 부탁드립니다.
[파이프라인 2회] 46자 / 위반 ['N4 길이 초과: 46자']
    초록 베이커리는 10월 9일 휴무입니다. 양해 부탁드리며 10월 10일 재개합니다.
[단일 3회] 48자 / 위반 ['N1 문장 수 위반: 마침표 1개', 'N4 길이 초과: 48자']
    초록 베이커리는 10월 9일 휴무입니다 양해 부탁드리며 10월 10일 영업 재개합니다.
[파이프라인 3회] 46자 / 위반 ['N4 길이 초과: 46자']
    초록 베이커리는 10월 9일 휴무입니다. 양해 부탁드리며 10월 10일 재개합니다.


**관찰 포인트**: 도메인이 바뀌어도 "작성 → 코드 검사 → 수정" 구조가 그대로 통했는가? 이 도메인에서 단일 에이전트가 주로 깬 규칙은 무엇인가?

## 확인 문제

1. 검사기를 LLM이 아닌 Python 함수로 만든 이유는 무엇인가?
2. 수정자에게 "기존 문장 안에 녹여 넣으라"고 지시하지 않으면 어떤 일이 생기는가?
3. 2-1에서 규칙을 3개로 줄였을 때 단일 에이전트는 어땠는가? 이것이 멀티 에이전트 선택에 주는 의미는?
4. 자기 검토(2-2)와 협업 파이프라인(1-4)의 차이는 무엇인가?
5. 수정 루프(2-6)의 `max_revision`은 왜 필요한가?
6. Part 3 결과를 근거로, 어떤 조건에서 멀티 에이전트(협업·루프)를 선택하겠는가?
7. 3-6에서 파이프라인이 N4(40자 이내)를 끝내 고치지 못한 이유와 대책은 무엇인가?

## 정리·회고

오늘 배운 것을 3줄로 정리해 본다.

1. 단일 에이전트가 규칙 8개에서 무너진 이유는 무엇이었는가?
2. 협업에서 "판정"과 "수정"을 누가 맡았고, 왜 그렇게 나눴는가?
3. 협업·루프의 품질 향상은 공짜였는가? 숫자로 답한다.

| 오늘 만든 것 | 다음에 쓰는 곳 |
| --- | --- |
| 검사기 `check_rules` | M2 Day 6 — FastAPI 서비스(상품 소개문 서비스)의 부품이 된다 |
| 작성자 → 검사기 → 수정자 | Day 2 역할·계약 설계, Day 5 피드백 루프 |
| 범용 파이프라인 `run_pipeline` | Day 2 — 역할과 주고받는 형식을 분리하는 출발점 |

작성한 요약과 오늘 코드를 커밋한다.

```powershell
git add .
git commit -m "Day1: 단일 vs 멀티 에이전트 비교 실험"
```

**내일 예고**: 오늘은 역할 2개를 함수로 하드코딩했다. Day 2에서는 역할을 **체계적으로 설계**한다 — 역할 분리 원칙, 업무 분할, 에이전트 사이의 데이터 계약(Pydantic).

실습문제 1

## 문제 1. 택배 도착 안내 문자 — 기본형


쇼핑몰이 고객에게 보내는 택배 도착 안내 문자를 쓴다.


| 규칙 | 내용 | 코드 판정 |
| --- | --- | --- |
| D1 | 정확히 3문장 (마침표 정확히 3개) | 마침표 개수 |
| D2 | "김하늘님"을 그대로 포함 | 포함 여부 |
| D3 | 도착일 "10월 12일"과 시간대 "오후 2~4시"를 그대로 포함 | 포함 여부 |
| D4 | 운송장 번호 "5821-3390-1147"을 그대로 포함 | 포함 여부 |
| D5 | 느낌표·물음표를 쓰지 않는다 | 포함 여부 |
| D6 | 쉼표를 쓰지 않는다 | 포함 여부 |
| D7 | 공백 포함 80자 이내 | `len()` |
| D8 | 마지막 문장은 따로 떨어진 한 문장 "감사합니다." | `endswith(". 감사합니다.")` |


**할 일**


1. 검사기 `check_delivery(text)` — 위반 목록을 반환한다. 위반이 없으면 `[]`. 아래 예문 `GOOD`·`BAD`로 먼저 시험한다.
2. 작성 지시문 `DELIVERY_PROMPT` — 주문 정보와 규칙 8개를 담는다.
3. 단일 `run_single_delivery()` — 작성자를 한 번 부른다. 반환: (글, 토큰, 시간)
4. 협업 `run_delivery_team()` — 작성 → 검사 → 위반이 있으면 수정자 1회 → 재검사. 반환: (최종 글, 최종 위반, 토큰, 시간)
5. 단일·협업을 각 3회 실행해 평균 위반 수와 평균 토큰을 표로 비교한다.


**확인 기준**: `GOOD`는 `[]`, `BAD`는 D1·D3·D5·D8 위반이 나온다. 비교표에서 협업의 평균 위반이 단일보다 같거나 적다.

In [72]:
order = {
    "name": "김하늘",
    "item": "무선 키보드",
    "date": "10월 12일",
    "time": "오후 2~4시",
    "tracking": "5821-3390-1147",
}


GOOD = "김하늘님 주문하신 무선 키보드가 10월 12일 오후 2~4시에 도착합니다. 운송장 번호는 5821-3390-1147입니다. 감사합니다."
BAD = "김하늘님 주문하신 무선 키보드가 내일 도착합니다! 운송장 번호는 5821-3390-1147입니다."
print(len(GOOD), "자")

75 자


In [73]:
# 규칙검사기
def check_delivery(text):
    """택배 도착 안내문의 D1~D8 규칙 위반 목록을 반환한다."""
    text = text.strip()
    violations = []

    if text.count(".") != 3:
        violations.append(f"D1 문장 수 위반: 마침표 {text.count('.')}개")
    if f'{order["name"]}님' not in text:
        violations.append("D2 고객명 누락")
    if order["date"] not in text or order["time"] not in text:
        violations.append("D3 도착일 또는 시간대 누락")
    if order["tracking"] not in text:
        violations.append("D4 운송장 번호 누락")
    marks = [mark for mark in ["!", "?"] if mark in text]
    if marks:
        violations.append(f"D5 금지 문장부호: {marks}")
    if "," in text:
        violations.append("D6 쉼표 사용")
    if len(text) > 80:
        violations.append(f"D7 길이 초과: {len(text)}자")
    if not text.endswith(". 감사합니다."):
        violations.append("D8 마지막 문장 위반")

    return violations


print("GOOD:", check_delivery(GOOD))
print("BAD:", check_delivery(BAD))
print("-" * 50)

# 지시문 + 시스템프롬프트
DELIVERY_PROMPT = f"""
당신은 온라인 쇼핑몰의 택배 도착 안내 문자 작성자입니다.

[주문 정보]
고객명: {order["name"]}님
주문 상품: {order["item"]}
도착일: {order["date"]}
도착 시간대: {order["time"]}
운송장 번호: {order["tracking"]}

[작성 규칙]
1. 정확히 3문장으로 쓰고 마침표를 정확히 3개 사용한다.
2. "{order["name"]}님"을 그대로 포함한다.
3. "{order["date"]}"과 "{order["time"]}"를 그대로 포함한다.
4. 운송장 번호 "{order["tracking"]}"을 그대로 포함한다.
5. 느낌표와 물음표를 쓰지 않는다.
6. 쉼표를 쓰지 않는다.
7. 공백을 포함해 80자 이내로 쓴다.
8. 마지막 문장은 따로 떨어진 한 문장 "감사합니다."로 쓴다.

안내 문자만 출력하세요. 규칙 설명이나 인사말은 출력하지 마세요.
""".strip()

# 단일 에이전트
def run_single_delivery():
    """작성자를 한 번 호출한다. 반환: (글, 토큰, 시간)"""
    start = time.perf_counter()
    response = llm_writer.invoke(DELIVERY_PROMPT)
    sec = time.perf_counter() - start
    return response.content.strip(), get_token_usage(response), sec

# 멀티 에이전트
def run_delivery_team():
    """작성 → 검사 → 필요 시 1회 수정 → 재검사한다. 반환: (최종 글, 최종 위반, 토큰, 시간)"""
    start = time.perf_counter()
    draft_response = llm_writer.invoke(DELIVERY_PROMPT)
    final = draft_response.content.strip()
    tokens = get_token_usage(draft_response)
    violations = check_delivery(final)

    if violations:
        violation_text = "\n".join(f"- {item}" for item in violations)
        fix_prompt = f"""
당신은 규칙을 정확히 적용하는 수정자입니다.

[원래 작성 지시]
{DELIVERY_PROMPT}

[초안]
{final}

[위반 목록]
{violation_text}

위반한 부분을 고치되 이미 지킨 부분은 가능한 한 유지하세요.
반드시 3문장과 80자 이내 규칙을 지키고 수정한 안내 문자만 출력하세요.
""".strip()
        fix_response = llm_fixer.invoke(fix_prompt)
        final = fix_response.content.strip()
        tokens += get_token_usage(fix_response)
        violations = check_delivery(final)

    sec = time.perf_counter() - start
    return final, violations, tokens, sec


delivery_records = []
for mode in ["단일", "협업"]:
    for i in range(3):
        if mode == "단일":
            text, tokens, sec = run_single_delivery()
            violations = check_delivery(text)
        else:
            text, violations, tokens, sec = run_delivery_team()

        delivery_records.append({
            "구조": mode,
            "위반 수": len(violations),
            "토큰": tokens,
            "시간(초)": sec,
        })
        print(f"[{mode} {i + 1}회] 위반 {violations} / 시간 {sec:.2f}초")
        print("   " , text)

delivery_result = (
    pd.DataFrame(delivery_records)
    .groupby("구조", sort=False)
    .agg(**{"평균 위반 수": ("위반 수", "mean"), "평균 토큰": ("토큰", "mean"), "평균 시간(초)": ("시간(초)", "mean")})
    .round(2)
)
delivery_result

GOOD: []
BAD: ['D1 문장 수 위반: 마침표 1개', 'D3 도착일 또는 시간대 누락', "D5 금지 문장부호: ['!']", 'D8 마지막 문장 위반']
--------------------------------------------------
[단일 1회] 위반 [] / 시간 1.76초
    김하늘님께서 주문하신 무선 키보드가 10월 12일 오후 2~4시에 도착합니다. 운송장 번호는 5821-3390-1147입니다. 감사합니다.
[단일 2회] 위반 [] / 시간 1.14초
    김하늘님께서 주문하신 무선 키보드가 10월 12일 오후 2~4시에 도착합니다. 운송장 번호는 5821-3390-1147입니다. 감사합니다.
[단일 3회] 위반 [] / 시간 1.16초
    김하늘님 주문하신 무선 키보드가 10월 12일 오후 2~4시에 도착합니다. 운송장 번호는 5821-3390-1147입니다. 감사합니다.
[협업 1회] 위반 [] / 시간 2.41초
    김하늘님 주문하신 무선 키보드가 10월 12일 오후 2~4시에 도착 예정입니다. 운송장 번호는 5821-3390-1147입니다. 감사합니다.
[협업 2회] 위반 [] / 시간 2.05초
    김하늘님 무선 키보드가 10월 12일 오후 2~4시에 도착합니다. 운송장 번호는 5821-3390-1147입니다. 감사합니다.
[협업 3회] 위반 [] / 시간 1.08초
    김하늘님께서 주문하신 무선 키보드는 10월 12일 오후 2~4시에 도착합니다. 운송장 번호는 5821-3390-1147입니다. 감사합니다.


,평균 위반 수,평균 토큰,평균 시간(초)
구조,,,
단일,0.0,304.33,1.35
협업,0.0,600.00,1.85


### 택배 도착 안내 에이전트 시퀀스 다이어그램

![택배 도착 안내 에이전트 시퀀스](assets/delivery-sequence.svg)

<details><summary>Mermaid 원본 코드</summary>

```mermaid
sequenceDiagram
    autonumber
    actor User as 실행자
    participant App as 택배 안내 프로그램
    participant Writer as 작성자 LLM
    participant Checker as 규칙 검사기
    participant Fixer as 수정자 LLM

    alt 단일 에이전트
        User->>App: run_single_delivery()
        App->>Writer: DELIVERY_PROMPT(order)
        Writer-->>App: 택배 안내문 + 토큰
        App-->>User: 글 + 토큰 + 실행 시간
        User->>Checker: check_delivery(글)
        Checker-->>User: 위반 목록
    else 협업 에이전트
        User->>App: run_delivery_team()
        App->>Writer: DELIVERY_PROMPT(order)
        Writer-->>App: 초안 + 토큰
        App->>Checker: check_delivery(초안)
        Checker-->>App: 위반 목록
        alt 위반이 있음
            App->>Fixer: 원래 지시 + 초안 + 위반 목록
            Fixer-->>App: 수정본 + 토큰
            App->>Checker: check_delivery(수정본)
            Checker-->>App: 최종 위반 목록
        else 위반이 없음
            Note over App: 초안을 최종본으로 사용
        end
        App-->>User: 최종 글 + 최종 위반 + 누적 토큰 + 실행 시간
    end
```

</details>

### 단일 택배 도착 안내 에이전트 구성도

![단일 택배 도착 안내 에이전트 구성도](assets/delivery-single-architecture.svg)

<details><summary>Mermaid 원본 코드</summary>

```mermaid
flowchart LR
    O[order 객체<br/>고객명 · 상품 · 도착일<br/>시간대 · 운송장] --> P[DELIVERY_PROMPT<br/>주문 정보 + D1~D8]
    P --> S[run_single_delivery]
    S --> W[작성자 LLM<br/>temperature 0.9]
    W --> R[안내문 · 토큰 · 실행 시간]
    R -. 외부 평가 .-> C[check_delivery<br/>D1~D8 규칙 검사]
    C --> V[위반 목록]
```

</details>

> 단일 방식은 작성자를 한 번만 호출하므로 빠르고 토큰 소모가 적지만 발견된 위반을 자동으로 수정하지는 않는다.

### 협업 택배 도착 안내 에이전트 구성도

![협업 택배 도착 안내 에이전트 구성도](assets/delivery-team-architecture.svg)

<details><summary>Mermaid 원본 코드</summary>

```mermaid
flowchart LR
    O[order 객체] --> P[DELIVERY_PROMPT<br/>주문 정보 + D1~D8]
    P --> T[run_delivery_team]
    T --> W[작성자 LLM<br/>초안 작성]
    W --> C1[check_delivery<br/>초안 검사]
    C1 --> D{D1~D8<br/>위반 있음?}
    D -- 아니오 --> F[초안을 최종본으로 선택]
    D -- 예 --> X[수정자 LLM<br/>temperature 0]
    X --> C2[check_delivery<br/>수정본 재검사]
    C2 --> F2[수정본을 최종본으로 선택]
    F --> R[최종 글 · 최종 위반<br/>누적 토큰 · 실행 시간]
    F2 --> R
```

</details>

> 협업 방식은 작성자와 수정자의 역할을 분리하고 코드 검사기를 중간에 두어 위반이 있을 때만 수정자를 호출한다.